# Strategy lab — desk board (signal → action → risk)

**Class:** risk-policy / MM playbook **sims** — not naked tradable alpha.

Loads Phase-4 crash/SSM/event features, maps Promote IDs → hypothetical desk actions, overlays tick/OB equity, and links the HL ETH paper harness day.

| SoT | Path |
|-----|------|
| Desk memo §7 | [`../../DESK_MEMO.md`](../../DESK_MEMO.md) |
| Promote rollup | [`../../CHAPTER_INDEX.md`](../../CHAPTER_INDEX.md) |
| Playbook | [`../../TRADING_APPLICATIONS.md`](../../TRADING_APPLICATIONS.md) |
| Lab doc | [`STRATEGY_LAB.md`](STRATEGY_LAB.md) |
| Paper harness | [`../paper_harness/`](../paper_harness/) |

**Friction:** 2.0 bps one-way · **Gate:** 10bps / i_c≥5 · **Venues:** HL+Deribit+Kraken.


## 0. Paths + Promote ID board

Cross-link Phase-4 Promotes to the executable overlays in this lab / paper harness.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

LAB = Path('.').resolve()
APP = LAB.parent
BOOK = APP.parent
OUT = LAB / 'out'
FIGS = OUT / 'figs'
PANEL = APP / 'out' / 'event_panel'
BOARD = APP / 'out' / 'applications_board.json'
PAPER = APP / 'paper_harness' / 'out'
SIG = APP / 'out' / 'signal_boards'

# Signal → hypothetical action → risk (Promote IDs from CHAPTER_INDEX / DESK_MEMO)
PROMOTE_BOARD = [
    {
        'id': 'risk.ssm_severity_gate_10bps',
        'signal': 'Gated SSM (|ΔP|≥10bps, i_c≥5)',
        'action': 'Intensity / ladder input only — never raw z*=6 counts',
        'risk': 'Ungated floods; median ΔP≈0',
        'overlay': 'all',
        'verdict': 'Promote',
    },
    {
        'id': 'vol.sigma_m_noise_floor_1bp',
        'signal': 'σ_m ≥ 1 bp floor',
        'action': 'Infra gate before any SSM flag',
        'risk': 'Floor-off → count flood',
        'overlay': 'detect',
        'verdict': 'Promote',
    },
    {
        'id': 'risk.ssm_zstar_scan_table',
        'signal': 'Within-gated z percentiles',
        'action': 'observe→widen→size_cap→halt (p25/p50/p75)',
        'risk': 'Absolute z∈{8,10,12} collapses on gated tape',
        'overlay': 'kill_ladder_maker',
        'verdict': 'Promote',
    },
    {
        'id': 'info.nanex_subset_of_ssm',
        'signal': 'Nanex∩SSM nest',
        'action': 'Temporary pull (~15s) / escalate',
        'risk': 'Rare; high precision (~0.90)',
        'overlay': 'nanex_temp_pull',
        'verdict': 'Promote-as-risk-policy',
    },
    {
        'id': 'info.crash_v_vs_continuation',
        'signal': 'V vs continuation @5s',
        'action': 'Stay wide until V confirm; do not blind-restore on cont',
        'risk': 'FP V-recovery → halt overkill',
        'overlay': 'v_restore_confirm',
        'verdict': 'Promote (med) MM',
    },
    {
        'id': 'frag.thin_venue_crash_excess',
        'signal': 'HL thin excess (crash−vol share)',
        'action': 'Dashboard strip; optional size-cap on HL',
        'risk': 'Hard size-cap Hold (Δ|mo| CI includes 0)',
        'overlay': 'hl_thin_size_cap',
        'verdict': 'Promote monitor',
    },
    {
        'id': 'frag.volume_herfindahl_3venue',
        'signal': 'H^v / FEI capacity',
        'action': 'Capacity regime dashboard (schedule Hold)',
        'risk': 'Spearman unstable OOS',
        'overlay': 'monitor',
        'verdict': 'Promote monitor',
    },
    {
        'id': 'info.vpin_x_size_severity',
        'signal': 'VPIN×logN interact',
        'action': 'Feature state — not live sizing',
        'risk': 'Not tradable; boot CI excludes 0 only as feature',
        'overlay': 'feature',
        'verdict': 'Promote (med) feature',
    },
]

s = json.loads((OUT / 'summary.json').read_text())
board = json.loads(BOARD.read_text()) if BOARD.is_file() else {}
panel_meta = json.loads((PANEL / 'panel_meta.json').read_text()) if (PANEL / 'panel_meta.json').is_file() else {}

print(f"cells_ok={s['n_cells_ok']} friction={s['friction_bps']} days={s['days']}")
print(f"panel n_rows={panel_meta.get('n_rows')} gate={panel_meta.get('gate')}")
print('cadence:', (s.get('cadence_honesty') or '')[:280], '…')
print()
pdf = pd.DataFrame(PROMOTE_BOARD)[['id', 'verdict', 'signal', 'action', 'overlay', 'risk']]
display(pdf)


## 1. Crash / SSM / event features (panel)

Load `applications/out/event_panel/` — gated events, tiers, V-labels, thin excess, H^v/FEI.


In [ ]:
rows = json.loads((PANEL / 'panel_rows.json').read_text())
cells = []
event_rows = []
for r in rows:
    cells.append({
        'day': r['day'], 'symbol': r['symbol'], 'venue': r['venue'],
        'complete': bool(r.get('complete')),
        'n_trades': r.get('n_trades'),
        'ssm_raw_n': r.get('ssm_raw_n'), 'ssm_10_n': r.get('ssm_10_n'),
        'nanex_n': r.get('nanex_n'), 'nanex_nested_n': r.get('nanex_nested_n'),
        'nanex_prec': r.get('nanex_ssm_precision'),
        'H_v': r.get('H_v'), 'FEI': r.get('FEI'),
        'thin_excess': r.get('thin_excess'),
        'crash_share': r.get('crash_share'),
    })
    ev = r.get('events') or {}
    n = len(ev.get('ts_end') or [])
    for i in range(n):
        event_rows.append({
            'day': r['day'], 'symbol': r['symbol'], 'venue': r['venue'],
            'tier': (ev.get('tier') or [None])[i] if i < len(ev.get('tier') or []) else None,
            'z_peak': (ev.get('z_peak') or [None])[i] if i < len(ev.get('z_peak') or []) else None,
            'dp_pct': (ev.get('dp_pct') or [None])[i] if i < len(ev.get('dp_pct') or []) else None,
            'i_c': (ev.get('i_c') or [None])[i] if i < len(ev.get('i_c') or []) else None,
            'intensity_60s': (ev.get('intensity_60s') or [None])[i] if i < len(ev.get('intensity_60s') or []) else None,
            'recovery_label': (ev.get('recovery_label') or [None])[i] if i < len(ev.get('recovery_label') or []) else None,
            'mo_5s': (ev.get('mo_5s') or [None])[i] if i < len(ev.get('mo_5s') or []) else None,
            'nanex_overlap': bool((ev.get('nanex_overlap') or [0])[i]) if i < len(ev.get('nanex_overlap') or []) else False,
        })

cdf = pd.DataFrame(cells)
edf = pd.DataFrame(event_rows)
print(f'panel cells={len(cdf)} gated_events={len(edf)} complete={int(cdf.complete.sum())}')
print('tier counts:', edf.tier.value_counts().to_dict() if len(edf) else {})
print('V-share:', float((edf.recovery_label == 'v_recovery').mean()) if len(edf) else float('nan'))
print('mean mo@5s (bps):', float(pd.to_numeric(edf.mo_5s, errors='coerce').mean()) if len(edf) else float('nan'))
display(cdf.sort_values(['day', 'symbol', 'venue']).head(12))

# Focus: HL ETH densest crash day from lab summary
focus = s.get('focus_cell') or {}
print('focus_cell:', focus)
fmask = (
    (edf.day == focus.get('day'))
    & (edf.symbol == focus.get('symbol'))
    & (edf.venue == focus.get('venue'))
)
focus_ev = edf[fmask].copy()
if len(focus_ev):
    display(Markdown(f"### Focus events — {focus.get('day')} {focus.get('venue')} {focus.get('symbol')} (n={len(focus_ev)})"))
    display(focus_ev[['tier', 'z_peak', 'dp_pct', 'i_c', 'intensity_60s', 'recovery_label', 'mo_5s', 'nanex_overlap']].head(20))


## 2. Signal → hypothetical action → risk (apps board)

Rollup from `applications_board.json` + Promote table above. Equity overlays are **sims**; event-study Promotes still stand in DESK_MEMO §7.


In [ ]:
def _get(d, *keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

app_rows = [
    {
        'package': 'kill_ladder',
        'promote_ids': 'risk.ssm_severity_gate_10bps · risk.ssm_zstar_scan_table · vol.sigma_m_noise_floor_1bp',
        'verdict': _get(board, 'kill_ladder', 'readiness'),
        'headline': f"Δ|mo|={_get(board,'kill_ladder','abs_mo_delta','delta'):.2f} "
                    f"CI=[{_get(board,'kill_ladder','abs_mo_delta','lo'):.2f},"
                    f"{_get(board,'kill_ladder','abs_mo_delta','hi'):.2f}]"
                    if _get(board, 'kill_ladder', 'abs_mo_delta') else 'n/a',
        'action': 'observe→widen→size_cap→halt on gated intensity',
        'risk': 'FP V-recovery among fire; venue-local only',
    },
    {
        'package': 'nanex_burst',
        'promote_ids': 'info.nanex_subset_of_ssm',
        'verdict': _get(board, 'nanex_burst', 'auto_pull_readiness'),
        'headline': f"prec={_get(board,'nanex_burst','precision'):.3f} "
                    f"Δ|ΔP|={_get(board,'nanex_burst','dp_effect','delta'):.3f}"
                    if _get(board, 'nanex_burst', 'precision') is not None else 'n/a',
        'action': 'nested temp pull on Nanex∩SSM',
        'risk': 'Low count; escalate not standalone alpha',
    },
    {
        'package': 'hl_thin_sor',
        'promote_ids': 'frag.thin_venue_crash_excess · frag.crash_venue_share',
        'verdict': _get(board, 'hl_thin_sor', 'readiness'),
        'headline': f"thin_excess={_get(board,'hl_thin_sor','thin_excess_mean','point'):.2f}"
                    if _get(board, 'hl_thin_sor', 'thin_excess_mean') else 'n/a',
        'action': 'HL strip yes; hard size-cap Hold',
        'risk': 'Δ|mo| size-cap CI includes 0',
    },
    {
        'package': 'mm_quoting (sibling)',
        'promote_ids': 'info.crash_v_vs_continuation',
        'verdict': 'Promote (med) MM',
        'headline': 'V mo@5s≈−16.5 / cont≈+12.1',
        'action': 'confirm-V restore / stay-wide on cont',
        'risk': 'Not fade-the-V; playbook only',
    },
]
display(pd.DataFrame(app_rows))

print('=== strategy_lab equity Δ vs baseline (pool) ===')
for name, block in (s.get('strategies') or {}).items():
    d = block.get('delta_vs_baseline') or {}
    pool = block.get('pool') or {}
    print(
        f"{name:22s} final_mean={pool.get('final_equity_bps', {}).get('mean', float('nan')):+.3f}  "
        f"Δ={d.get('delta_mean', float('nan')):+.3f} "
        f"CI=[{d.get('lo', float('nan')):+.3f},{d.get('hi', float('nan')):+.3f}] "
        f"fills={pool.get('n_fills_total')}"
    )
print()
print('verdict:', json.dumps(s.get('verdict'), indent=2))


## 3. Equity / drawdown / regimes / book (lab figs)

PNG artifacts under `out/figs/`. Interpret vs `baseline_maker`; CI including 0 → Hold as live PnL claim (may still be Promote-as-risk-policy from event-study).


In [ ]:
preferred = [
    'equity_curves_focus.png',
    'drawdown_baseline_vs_ladder.png',
    'price_regimes_fills.png',
    'inventory_path.png',
    'delta_vs_baseline.png',
    'early_late_split.png',
    'book_depth_events.png',
    'book_source_counts.png',
]
shown = set()
for name in preferred:
    p = FIGS / name
    if p.is_file():
        display(Markdown(f'### `{name}`'))
        display(Image(filename=str(p)))
        shown.add(name)
for p in sorted(FIGS.glob('*.png')):
    if p.name not in shown:
        display(Markdown(f'### `{p.name}`'))
        display(Image(filename=str(p)))


## 4. Paper harness — complete HL ETH day

Detect → kill-ladder → shadow fills → daily `RISK_REPORT`. Reproducible CLI:

```bash
cd applications/paper_harness
python3 run_paper_day.py --day 2026-09-04
```


In [ ]:
paper_dirs = sorted(PAPER.glob('*_hyperliquid_ETH'))
if not paper_dirs:
    print('No paper_harness HL ETH out yet — run: python3 ../paper_harness/run_paper_day.py --day 2026-09-04')
else:
    day_dir = paper_dirs[0]
    # prefer 2026-09-04 if present
    for d in paper_dirs:
        if d.name.startswith('2026-09-04'):
            day_dir = d
            break
    ps = json.loads((day_dir / 'summary.json').read_text())
    sh = ps.get('shadow') or {}
    dlt = sh.get('delta_vs_baseline') or {}
    print(f"paper day={ps.get('day')} complete={ps.get('complete')} trades={ps.get('n_trades')}")
    print(f"SSM raw/gated={ps.get('ssm_raw_n')}/{ps.get('ssm_10_n')}  "
          f"fire/observe={ps.get('n_fire')}/{ps.get('n_observe')}  "
          f"nanex_esc={ps.get('n_nanex_escalate')}")
    print(f"mo@5s fire={ps.get('mo5_fire_mean')}  FP_V@fire={ps.get('fp_v_recovery_rate_among_fire')}")
    print(f"fill_model={sh.get('fill_model')}  Δequity={dlt.get('delta_equity_bps')}  "
          f"Δfills={dlt.get('delta_n_fills')}")
    print(f"ladder fills_by_regime={(sh.get('kill_ladder') or {}).get('fills_by_regime')}")
    print()
    print('Promote IDs wired:')
    for p in ps.get('promote_ids') or []:
        print(f"  {p['id']:32s}  {p['verdict']:28s}  {p['action']}")
    md = day_dir / 'RISK_REPORT.md'
    if md.is_file():
        display(Markdown(f"### RISK_REPORT — `{day_dir.name}`\n\n" + md.read_text()[:2500] + ('…' if len(md.read_text()) > 2500 else '')))
    fig_dir = day_dir / 'figs'
    for fig in sorted(fig_dir.glob('*.png')):
        display(Markdown(f'#### paper `{fig.name}`'))
        display(Image(filename=str(fig)))


## 5. Signal-board companion figs (optional)

Shared intensity / thin-excess / Nanex boards — do not overwrite strategy equity figs.


In [ ]:
sig_figs = [
    'fig_gated_intensity_ts.png',
    'fig_zstar_path_ts.png',
    'fig_nanex_ssm_ts.png',
    'fig_thin_excess_ts.png',
    'fig_event_heatmaps.png',
]
sf = SIG / 'figs'
if sf.is_dir():
    for name in sig_figs:
        p = sf / name
        if p.is_file():
            display(Markdown(f'### signal `{name}`'))
            display(Image(filename=str(p)))
        else:
            print('missing', p)
else:
    print('signal_boards figs not present at', sf)


## How to interpret

1. **Promote IDs** — risk-policy / monitor / MM playbook / feature only; **0 tradable**.
2. **Signal → action → risk** — every overlay must name the Promote it implements and its known falsifier.
3. **Equity (bps)** — synthetic touch-maker after friction; compare to `baseline_maker`.
4. **Δ vs baseline CI** — includes 0 → Hold as live PnL; event-study Promote-as-risk-policy may still stand.
5. **Paper harness** — one complete HL ETH UTC day; tape-proxy fill model when warehouse BBO is stale (>5s).
6. **Cadence** — Phase-4 days usually lack collector TOB; do not read curves as sub-second queue edge.

### Re-run lab

```bash
cd applications/strategy_lab
python3 exp_strategy_lab.py --workers 14
python3 exp_strategy_lab.py --hl-only --workers 10
```
